# NLP Early Warning Framework

## Overview

This notebook explores building the rule-based early-warning system using the multi-label case table plus the text sidecar.
It is not a supervised prediction model.

The goal is to help identify make/model/component cohorts that look like emerging issues based on:
- complaint count growth
- unusual month-over-month behavior compared with prior months
- broad severity share
- complaint text clues

Inputs:
- `data/processed/odi_component_multilabel_cases.parquet`
- `data/processed/odi_component_text_sidecar.parquet`

Outputs:
- `data/outputs/nlp_early_warning_watchlist.csv`
- `data/outputs/nlp_early_warning_terms.csv`

## Setup

This cell sets imports, project paths, and helper constants.
You should be able to run this notebook from VS Code or Colab as long as the repo files are available.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

RANDOM_SEED = 42

# Resolve repo paths so the notebook works from repo root or notebooks/
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'

# Current exploratory inputs from consolidated preprocessing
MULTI_PATH = PROCESSED_DATA_DIR / 'odi_component_multilabel_cases.parquet'
SIDECAR_PATH = PROCESSED_DATA_DIR / 'odi_component_text_sidecar.parquet'

# Optional exploratory outputs for watchlist review
WATCHLIST_OUTPUT_PATH = OUTPUTS_DIR / 'nlp_early_warning_watchlist.csv'
TERMS_OUTPUT_PATH = OUTPUTS_DIR / 'nlp_early_warning_terms.csv'

print(f"Project root: {PROJECT_ROOT}")
print(f"Multi-label input: {MULTI_PATH}")
print(f"Text sidecar input: {SIDECAR_PATH}")
print(f"Watchlist output: {WATCHLIST_OUTPUT_PATH}")
print(f"Terms output: {TERMS_OUTPUT_PATH}")

Project root: c:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics
Multi-label input: c:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\processed\odi_component_multilabel_cases.parquet
Text sidecar input: c:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\processed\odi_component_text_sidecar.parquet
Watchlist output: c:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\outputs\nlp_early_warning_watchlist.csv
Terms output: c:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\outputs\nlp_early_warning_terms.csv


## Data Loading

Both inputs come from the current consolidated preprocessing stage. The multi-label table keeps the structured cohort fields, and the text sidecar keeps narrative text separate until an exploration notebook wants to use it.
Load them here and join on `odino`.

In [2]:
if not MULTI_PATH.exists():
    raise FileNotFoundError(f"Missing input file: {MULTI_PATH}")
if not SIDECAR_PATH.exists():
    raise FileNotFoundError(f"Missing input file: {SIDECAR_PATH}")

multi_df = pd.read_parquet(MULTI_PATH)
sidecar_df = pd.read_parquet(SIDECAR_PATH)

join_cols = [
    'odino',
    'cdescr',
    'cdescr_model_text',
    'cdescr_missing_flag',
    'cdescr_placeholder_flag',
    'cdescr_char_len',
    'cdescr_word_count'
]
sidecar_use = sidecar_df[join_cols].copy()

base_df = multi_df.merge(sidecar_use, on='odino', how='left', validate='one_to_one')
base_df['ldate'] = pd.to_datetime(base_df['ldate'])
base_df['month'] = base_df['ldate'].dt.to_period('M')

text_candidate = base_df['cdescr_model_text'].fillna('').astype(str).str.strip()
fallback_text = base_df['cdescr'].fillna('').astype(str).str.strip()
base_df['text_for_nlp'] = np.where(text_candidate.ne(''), text_candidate, fallback_text)

print(f"Base rows: {len(base_df):,}")
print(f"Date range: {base_df['ldate'].min().date()} to {base_df['ldate'].max().date()}")
base_df[['odino', 'month', 'mfr_name', 'maketxt', 'modeltxt', 'component_groups', 'text_for_nlp']].head(3)

Base rows: 346,590
Date range: 2020-01-01 to 2026-02-19


,odino,month,mfr_name,maketxt,modeltxt,component_groups,text_for_nlp
0,10816121,2020-10,"GENERAL MOTORS, LLC",PONTIAC,VIBE,AIR BAGS,2007 PONTIAC VIBE. CONSUMER WRITES IN REGARDS ...
1,11289434,2020-01,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,CR-V,STEERING,2019 HONDA CR-V. CONSUMER WRITES IN REGARDS TO...
2,11289436,2020-01,HYUNDAI MOTOR AMERICA,HYUNDAI,SONATA,VEHICLE SPEED CONTROL,2018 HYUNDAI SONATA. CONSUMER WRITES IN REGARD...


## Cohort Table

Create complaint-component rows by exploding `component_groups`.
Then aggregate to monthly cohort rows by:
- `mfr_name`
- `maketxt`
- `modeltxt`
- `yeartxt`
- `component_group`
- `month`

In [3]:
# TODO: Choose the cohort definition. Good starter value: ['mfr_name', 'maketxt', 'modeltxt', 'yeartxt', 'component_group']
cohort_key_cols = ['mfr_name', 'maketxt', 'modeltxt', 'yeartxt', 'component_group']

if cohort_key_cols is None:
    raise ValueError('TODO: Set cohort_key_cols before building monthly cohorts.')


rows_df = base_df.copy()
rows_df['component_group'] = (
    rows_df['component_groups']
    .fillna('')
    .astype(str)
    .str.split('|')
)
rows_df = rows_df.explode('component_group', ignore_index=True)
rows_df['component_group'] = rows_df['component_group'].fillna('').astype(str).str.strip()
rows_df = rows_df.loc[rows_df['component_group'].ne('')].copy()

monthly_df = (
    rows_df
    .groupby(cohort_key_cols + ['month'], as_index=False)
    .agg(
        complaint_count=('odino', 'nunique'),
        broad_severity_count=('severity_broad_flag', 'sum'),
        primary_severity_count=('severity_primary_flag', 'sum'),
        unique_states=('state', 'nunique'),
        avg_text_len=('cdescr_char_len', 'mean')
    )
)

monthly_df['broad_severity_rate'] = monthly_df['broad_severity_count'] / monthly_df['complaint_count'].clip(lower=1)
monthly_df['primary_severity_rate'] = monthly_df['primary_severity_count'] / monthly_df['complaint_count'].clip(lower=1)

monthly_df = monthly_df.sort_values(cohort_key_cols + ['month']).reset_index(drop=True)

print(f"Exploded complaint-component rows: {len(rows_df):,}")
print(f"Monthly cohort rows: {len(monthly_df):,}")
monthly_df.head(5)

Exploded complaint-component rows: 471,961
Monthly cohort rows: 297,663


,mfr_name,maketxt,modeltxt,yeartxt,component_group,month,complaint_count,broad_severity_count,primary_severity_count,unique_states,avg_text_len,broad_severity_rate,primary_severity_rate
0,"4-STAR TRAILERS, INC.",4-STAR,HORSE TRAILER,2019,POWER TRAIN,2022-03,1,0,0,1,371.0,0.0,0.0
1,"4-STAR TRAILERS, INC.",4-STAR,HORSE TRAILER,2019,SUSPENSION,2025-06,1,0,0,1,658.0,0.0,0.0
2,ABS REMORQUES,ABS REMORQUES,LRC,2023,ENGINE / COOLING,2026-01,1,1,1,1,1063.0,1.0,1.0
3,"AIRSTREAM, INC.",AIRSTREAM,ATLAS,2020,STRUCTURE,2025-12,1,0,0,1,1253.0,0.0,0.0
4,"AIRSTREAM, INC.",AIRSTREAM,ATLAS,2023,FORWARD COLLISION AVOIDANCE,2024-03,1,0,0,1,503.0,0.0,0.0


## Leakage-Safe Features

Build prior/rolling features using only earlier months for each cohort.
No future month values are used to build the current month score.

Features added:
- `prior_3m_mean_count`
- `prior_6m_mean_count`
- `prior_6m_std_count`
- `growth_ratio`
- `z_score`


In [ ]:
# TODO: Choose how many earlier months should define recent and longer-history baselines. Good starter values are 3 and 6.
SHORT_WINDOW_MONTHS = 3
LONG_WINDOW_MONTHS = 6

if SHORT_WINDOW_MONTHS is None or LONG_WINDOW_MONTHS is None:
    raise ValueError('TODO: Set SHORT_WINDOW_MONTHS and LONG_WINDOW_MONTHS before creating prior-month features.')
if SHORT_WINDOW_MONTHS <= 0 or LONG_WINDOW_MONTHS <= 0:
    raise ValueError('Window sizes must be positive integers.')

feature_df = monthly_df.sort_values(cohort_key_cols + ['month']).reset_index(drop=True).copy()

# Each prior feature below uses only earlier rows within the same cohort.
cohort_grouped = feature_df.groupby(cohort_key_cols, sort=False, dropna=False)
feature_df['months_with_history'] = cohort_grouped.cumcount()

max_window = max(SHORT_WINDOW_MONTHS, LONG_WINDOW_MONTHS)
lag_cols = []
for lag in range(1, max_window + 1):
    col = f'_prior_count_lag_{lag}'
    feature_df[col] = cohort_grouped['complaint_count'].shift(lag)
    lag_cols.append(col)

short_lag_cols = lag_cols[:SHORT_WINDOW_MONTHS]
long_lag_cols = lag_cols[:LONG_WINDOW_MONTHS]

feature_df['prior_3m_mean_count'] = feature_df[short_lag_cols].mean(axis=1, skipna=True)
feature_df['prior_6m_mean_count'] = feature_df[long_lag_cols].mean(axis=1, skipna=True)
prior_6m_non_missing = feature_df[long_lag_cols].notna().sum(axis=1)
feature_df['prior_6m_std_count'] = feature_df[long_lag_cols].std(axis=1, skipna=True, ddof=0).where(prior_6m_non_missing >= 2)

feature_df['growth_ratio'] = feature_df['complaint_count'] / feature_df['prior_6m_mean_count'].replace(0, np.nan)
denom = feature_df['prior_6m_std_count'].replace(0, np.nan)
feature_df['z_score'] = (feature_df['complaint_count'] - feature_df['prior_6m_mean_count']) / denom
feature_df['z_score'] = feature_df['z_score'].replace([np.inf, -np.inf], np.nan)
feature_df = feature_df.drop(columns=lag_cols)

feature_df[['complaint_count', 'prior_3m_mean_count', 'prior_6m_mean_count', 'prior_6m_std_count', 'growth_ratio', 'z_score']].describe().T

,count,mean,std,min,25%,50%,75%,max
complaint_count,297663.0,1.584090,2.215490,1.000000,1.000000,1.000000,1.000000,197.000000
prior_3m_mean_count,231141.0,1.727578,2.140099,1.000000,1.000000,1.000000,1.666667,99.000000
prior_6m_mean_count,231141.0,1.728521,1.991570,1.000000,1.000000,1.166667,1.666667,88.166667
prior_6m_std_count,195362.0,0.694091,1.403047,0.000000,0.000000,0.471405,0.816497,72.897188
growth_ratio,231141.0,1.056656,0.817630,0.024896,0.750000,1.000000,1.000000,147.750000
z_score,127861.0,0.003180,2.494151,-35.000000,-0.780869,-0.447214,0.353553,415.071681


## Watchlist Score

Create a fixed rule-based score using:
- positive z-score
- positive growth ratio above 1.0
- complaint volume
- broad severity rate

Then keep rows where:
- at least one prior month exists (`months_with_history >= 1`)
- min complaints and top kept complaints >= set values

In [5]:
# TODO: Choose watchlist settings. Good starter values are shown in the comments below.
WATCHLIST_WEIGHTS = {'z': 0.35, 'growth': 0.30, 'volume': 0.20, 'severity': 0.15}  # Try {'z': 0.35, 'growth': 0.30, 'volume': 0.20, 'severity': 0.15} and adjust based on results
MIN_COMPLAINTS_PER_COHORT_MONTH = 3  # Try 3 and adjust based on resulting watchlist sizes
TOP_N_PER_MONTH = 25  # Try 25 and adjust based on resulting watchlist sizes

if WATCHLIST_WEIGHTS is None or MIN_COMPLAINTS_PER_COHORT_MONTH is None or TOP_N_PER_MONTH is None:
    raise ValueError('TODO: Fill in WATCHLIST_WEIGHTS, MIN_COMPLAINTS_PER_COHORT_MONTH, and TOP_N_PER_MONTH before scoring the watchlist.')

required_weight_keys = {'z', 'growth', 'volume', 'severity'}
if set(WATCHLIST_WEIGHTS) != required_weight_keys:
    raise ValueError(f'WATCHLIST_WEIGHTS must contain exactly these keys: {sorted(required_weight_keys)}')

score_df = feature_df.copy()

score_df['z_score_pos'] = score_df['z_score'].clip(lower=0)
score_df['growth_ratio_pos'] = (score_df['growth_ratio'] - 1.0).clip(lower=0)
score_df['volume_score_raw'] = np.log1p(score_df['complaint_count'])
score_df['severity_score_raw'] = score_df['broad_severity_rate'].fillna(0)

z_scale = score_df['z_score_pos'].quantile(0.95)
g_scale = score_df['growth_ratio_pos'].quantile(0.95)
v_scale = score_df['volume_score_raw'].max()

score_df['z_score_norm'] = score_df['z_score_pos'] / z_scale if pd.notna(z_scale) and z_scale > 0 else 0
score_df['growth_ratio_norm'] = score_df['growth_ratio_pos'] / g_scale if pd.notna(g_scale) and g_scale > 0 else 0
score_df['volume_norm'] = score_df['volume_score_raw'] / v_scale if pd.notna(v_scale) and v_scale > 0 else 0
score_df['severity_norm'] = score_df['severity_score_raw'].clip(lower=0, upper=1)

for col in ['z_score_norm', 'growth_ratio_norm', 'volume_norm', 'severity_norm']:
    score_df[col] = score_df[col].clip(lower=0, upper=1)

score_df['watchlist_score'] = (
    WATCHLIST_WEIGHTS['z'] * score_df['z_score_norm'] +
    WATCHLIST_WEIGHTS['growth'] * score_df['growth_ratio_norm'] +
    WATCHLIST_WEIGHTS['volume'] * score_df['volume_norm'] +
    WATCHLIST_WEIGHTS['severity'] * score_df['severity_norm']
)

filtered_df = score_df.loc[
    (score_df['complaint_count'] >= MIN_COMPLAINTS_PER_COHORT_MONTH) &
    (score_df['months_with_history'] >= 1)
].copy()

filtered_df['month_rank'] = (
    filtered_df
    .groupby('month')['watchlist_score']
    .rank(method='first', ascending=False)
)

watchlist_df = (
    filtered_df
    .loc[filtered_df['month_rank'] <= TOP_N_PER_MONTH]
    .sort_values(['month', 'month_rank', 'watchlist_score'], ascending=[True, True, False])
    .reset_index(drop=True)
)

WATCHLIST_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
watchlist_df.to_csv(WATCHLIST_OUTPUT_PATH, index=False)

print(f"Watchlist rows: {len(watchlist_df):,}")
print(f"Saved: {WATCHLIST_OUTPUT_PATH}")
watchlist_df.head(10)

Watchlist rows: 1,800
Saved: c:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\outputs\nlp_early_warning_watchlist.csv


,mfr_name,maketxt,modeltxt,yeartxt,component_group,month,complaint_count,broad_severity_count,primary_severity_count,unique_states,...,z_score_pos,growth_ratio_pos,volume_score_raw,severity_score_raw,z_score_norm,growth_ratio_norm,volume_norm,severity_norm,watchlist_score,month_rank
0,HYUNDAI MOTOR AMERICA,HYUNDAI,ELANTRA,2017,ENGINE / COOLING,2020-03,4,2,1,4,...,5.0,1.666667,1.609438,0.5,1.000000,1.0,0.304341,0.5,0.785868,1.0
1,"SUBARU OF AMERICA, INC.",SUBARU,OUTBACK,2020,VISIBILITY / WIPER,2020-03,16,0,0,13,...,10.0,1.666667,2.833213,0.0,1.000000,1.0,0.535755,0.0,0.757151,2.0
2,FORD MOTOR COMPANY,FORD,FUSION,2017,ENGINE / COOLING,2020-03,3,1,1,3,...,3.0,1.000000,1.386294,0.333333,1.000000,1.0,0.262145,0.333333,0.752429,3.0
3,"SUBARU OF AMERICA, INC.",SUBARU,OUTBACK,2013,ENGINE / COOLING,2020-03,3,1,0,3,...,3.0,1.000000,1.386294,0.333333,1.000000,1.0,0.262145,0.333333,0.752429,4.0
4,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,ACCORD,2000,AIR BAGS,2020-03,8,1,1,6,...,13.0,4.333333,2.197225,0.125,1.000000,1.0,0.415490,0.125,0.751848,5.0
5,TOYOTA MOTOR CORPORATION,TOYOTA,PRIUS,2010,SERVICE BRAKES,2020-03,7,1,1,4,...,7.0,1.000000,2.079442,0.142857,1.000000,1.0,0.393218,0.142857,0.750072,6.0
6,"CHRYSLER (FCA US, LLC)",JEEP,CHEROKEE,2016,ELECTRICAL SYSTEM,2020-03,11,0,0,10,...,6.0,1.200000,2.484907,0.0,1.000000,1.0,0.469891,0.0,0.743978,7.0
7,"CHRYSLER (FCA US, LLC)",JEEP,CHEROKEE,2018,POWER TRAIN,2020-03,11,0,0,8,...,7.0,1.750000,2.484907,0.0,1.000000,1.0,0.469891,0.0,0.743978,8.0
8,FORD MOTOR COMPANY,FORD,F-150,2012,POWER TRAIN,2020-03,5,0,0,4,...,5.0,1.000000,1.791759,0.0,1.000000,1.0,0.338818,0.0,0.717764,9.0
9,HYUNDAI MOTOR AMERICA,HYUNDAI,SONATA,2013,ENGINE / COOLING,2020-03,9,2,0,6,...,2.5,1.250000,2.302585,0.222222,0.842516,1.0,0.435414,0.222222,0.715297,10.0


## Text Clues

For top watchlist cohort-month rows, extract:
- a few sample complaint narratives
- top TF-IDF terms/phrases

These are clues for review.

In [ ]:
latest_month = watchlist_df['month'].max() if len(watchlist_df) else None
if latest_month is None:
    selected_watchlist = watchlist_df.copy()
else:
    selected_watchlist = watchlist_df.loc[watchlist_df['month'].eq(latest_month)].sort_values('month_rank').head(5).copy()

display_cols = ['month', 'month_rank', *cohort_key_cols, 'complaint_count', 'watchlist_score']
missing_display_cols = [col for col in display_cols if col not in selected_watchlist.columns]
if missing_display_cols:
    raise KeyError(f'Missing expected watchlist columns: {missing_display_cols}')

selected_watchlist[display_cols].head(10)

,month,month_rank,mfr_name,maketxt,modeltxt,yeartxt,component_group,complaint_count,watchlist_score
1775,2026-02,1.0,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,3,0.802429
1776,2026-02,2.0,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,CR-V,2020,ELECTRICAL SYSTEM,3,0.802429
1777,2026-02,3.0,FORD MOTOR COMPANY,FORD,F-150,2017,POWER TRAIN,51,0.802376
1778,2026-02,4.0,FORD MOTOR COMPANY,FORD,F-150,2016,POWER TRAIN,38,0.788554
1779,2026-02,5.0,FORD MOTOR COMPANY,FORD,F-150,2015,POWER TRAIN,34,0.784462


In [7]:
def top_terms_for_docs(text_series, top_n = 10):
    docs = text_series.fillna('').astype(str).str.strip()
    docs = docs.loc[docs.ne('')]
    if docs.empty:
        return pd.DataFrame(columns=['term', 'tfidf_score'])

    vectorizer = TfidfVectorizer(
        lowercase=True,
        analyzer='word',
        ngram_range=(1, 2),
        min_df=1,
        max_features=2000
    )
    mat = vectorizer.fit_transform(docs)
    avg_scores = np.asarray(mat.mean(axis=0)).ravel()
    terms = np.array(vectorizer.get_feature_names_out())
    top_idx = np.argsort(avg_scores)[::-1][:top_n]
    return pd.DataFrame({'term': terms[top_idx], 'tfidf_score': avg_scores[top_idx]})

term_rows = []
example_rows = []

for _, row in selected_watchlist.iterrows():
    mask = rows_df['month'].eq(row['month'])
    for col in cohort_key_cols:
        mask &= rows_df[col].eq(row[col])
    cohort_rows = rows_df.loc[mask].copy()

    cohort_identity = {col: row[col] for col in cohort_key_cols}

    sample_text = (
        cohort_rows['text_for_nlp']
        .fillna('')
        .astype(str)
        .str.strip()
        .loc[lambda s: s.ne('')]
        .drop_duplicates()
        .head(3)
    )
    for idx, txt in enumerate(sample_text, start=1):
        example_rows.append({
            'month': str(row['month']),
            **cohort_identity,
            'example_index': idx,
            'example_text': txt
        })

    terms_df = top_terms_for_docs(cohort_rows['text_for_nlp'], top_n=10)
    for _, trow in terms_df.iterrows():
        term_rows.append({
            'month': str(row['month']),
            **cohort_identity,
            'watchlist_score': row['watchlist_score'],
            'term': trow['term'],
            'tfidf_score': float(trow['tfidf_score'])
        })

terms_df = pd.DataFrame(term_rows)
examples_df = pd.DataFrame(example_rows)

TERMS_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
terms_df.to_csv(TERMS_OUTPUT_PATH, index=False)

print(f"Term rows: {len(terms_df):,}")
print(f"Saved: {TERMS_OUTPUT_PATH}")
terms_df.head(15)

Term rows: 50
Saved: c:\Users\davis\OneDrive\Documents\VS Code Repos\NHTSA-ODI-Complaint-Analytics\data\outputs\nlp_early_warning_terms.csv


,month,mfr_name,maketxt,modeltxt,yeartxt,component_group,watchlist_score,term,tfidf_score
0,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,0.802429,the,0.429932
1,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,0.802429,was,0.211587
2,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,0.802429,vehicle,0.159457
3,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,0.802429,to,0.129794
4,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,0.802429,the vehicle,0.125058
5,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,0.802429,not go,0.105548
6,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,0.802429,vehicle will,0.105548
7,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,0.802429,my,0.105548
8,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,0.802429,my vehicle,0.105548
9,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,0.802429,go,0.105548


## Results

Quick summary tables for the exploratory watchlist outputs and text examples.

In [8]:
monthly_summary = (
    watchlist_df
    .groupby('month', as_index=False)
    .agg(
        cohort_rows=('watchlist_score', 'count'),
        avg_watchlist_score=('watchlist_score', 'mean'),
        max_watchlist_score=('watchlist_score', 'max'),
        avg_complaint_count=('complaint_count', 'mean')
    )
)

print('Monthly watchlist summary:')
display(monthly_summary.tail(12))

print('Example complaint text snippets for selected cohorts:')
display(examples_df.head(15))

Monthly watchlist summary:


,month,cohort_rows,avg_watchlist_score,max_watchlist_score,avg_complaint_count
60,2025-03,25,0.783329,0.823593,19.28
61,2025-04,25,0.779511,0.852429,11.72
62,2025-05,25,0.765915,0.852429,9.36
63,2025-06,25,0.766559,0.852429,15.52
64,2025-07,25,0.764509,0.793966,12.52
65,2025-08,25,0.763019,0.807764,7.52
66,2025-09,25,0.76458,0.805394,12.72
67,2025-10,25,0.782485,0.852429,28.80
68,2025-11,25,0.76145,0.852429,7.32
69,2025-12,25,0.771393,0.823368,15.84


Example complaint text snippets for selected cohorts:


,month,mfr_name,maketxt,modeltxt,yeartxt,component_group,example_index,example_text
0,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,1,My vehicle will not go in reverse.
1,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,2,The contact owns a 2021 Chevrolet Suburban. Th...
2,2026-02,"GENERAL MOTORS, LLC",CHEVROLET,SUBURBAN,2021,POWER TRAIN,3,The contact owns a 2021 Chevrolet Suburban. Th...
3,2026-02,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,CR-V,2020,ELECTRICAL SYSTEM,1,Vehicle was parked on a slight incline with tr...
4,2026-02,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,CR-V,2020,ELECTRICAL SYSTEM,2,"While operating a 2020 Honda CR-V Hybrid, the ..."
5,2026-02,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,CR-V,2020,ELECTRICAL SYSTEM,3,I was driving the vehicle when all of a sudden...
6,2026-02,FORD MOTOR COMPANY,FORD,F-150,2017,POWER TRAIN,1,My 2017 ford f150 6r80 transmission lead frame...
7,2026-02,FORD MOTOR COMPANY,FORD,F-150,2017,POWER TRAIN,2,Frequently I will be driving at highway speeds...
8,2026-02,FORD MOTOR COMPANY,FORD,F-150,2017,POWER TRAIN,3,Transmission Valve Body and Lead Frame. With j...
9,2026-02,FORD MOTOR COMPANY,FORD,F-150,2016,POWER TRAIN,1,The contact owns a 2016 Ford F-150. The contac...


## Next TODOs / Extension Ideas

Use this section as a checklist for improving the early-warning notebook after the starter watchlist runs.

Suggested next steps:
- Change `cohort_key_cols` and compare how the watchlist changes. A more specific cohort like make/model/year/component is precise but can be noisy; a broader cohort like make/model/component is more stable.
- Try different `SHORT_WINDOW_MONTHS` and `LONG_WINDOW_MONTHS` values. Short windows react faster, while longer windows are less sensitive to one unusual month.
- Tune `WATCHLIST_WEIGHTS`, `MIN_COMPLAINTS_PER_COHORT_MONTH`, and `TOP_N_PER_MONTH` so the output is small enough for a person to review.
- Add a repeated-signal summary that counts which cohorts appear in the monthly top list more than once.
- Add a chart of complaint counts over time for a selected cohort so the score can be checked visually.
- Add more text clues, such as the top repeated phrases for each selected cohort or a few representative complaint narratives.
- Add a manual review column to the saved CSV, such as `review_notes` or `looks_actionable`, if the team wants to inspect rows together.
- Document false positives. If a row looks like noise, write down why so the score can be improved later.
- If this watchlist design stabilizes later, only the settled rules should move into `src`; the notebook should keep the exploration paths visible.

A high watchlist score means a cohort deserves a closer look.